# Graph Analizi – Bölüm 2

Bu notebook, Custom-WikiCS graf üzerinde aşağıdaki analizleri gerçekleştirir:

1. **Connected Component** sayısı
2. **Centrality ölçümleri**: Betweenness, Closeness, Degree, PageRank
   - Min / Max değerler
   - Dağılım histogramları
3. **Community Detection**: Louvain ve Infomap algoritmaları
   - Modularity
   - Community sayısı
   - Min / Max / Mean / Median community boyutu, std sapma
   - Community boyut dağılımı histogramı

Ağır hesaplamalar `./cache/` klasörüne kaydedilir, sonraki çalıştırmalarda cache'ten okunur.

## 1. Veri Yükleme ve Graf Oluşturma

In [ ]:
import json, os, pickle
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter

CACHE_DIR = './cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# --- Veri yükle ---
with open('./data/data-embeddings.json', 'r') as f:
    data = json.load(f)
print(f'Toplam node sayısı: {len(data["nodes"])}')

# --- Graf oluştur ---
node_ids = {n['id'] for n in data['nodes']}
G = nx.DiGraph()
for n in data['nodes']:
    G.add_node(n['id'], title=n['title'], label=n.get('label', 'Unknown'))
for n in data['nodes']:
    src = n['id']
    for tgt in n.get('outlinks', []):
        if tgt in node_ids:
            G.add_edge(src, tgt)

# Undirected versiyon (birçok metrik için gerekli)
G_und = G.to_undirected()

print(f'Node sayısı: {G.number_of_nodes()}')
print(f'Edge sayısı: {G.number_of_edges()}')
print(f'Undirected edge sayısı: {G_und.number_of_edges()}')

## 2. Connected Component Sayısı

In [ ]:
# Directed graf için weakly ve strongly connected component sayıları
n_weakly  = nx.number_weakly_connected_components(G)
n_strongly = nx.number_strongly_connected_components(G)

print(f'Weakly Connected Component sayısı:  {n_weakly}')
print(f'Strongly Connected Component sayısı: {n_strongly}')

# En büyük weakly connected component boyutu
largest_wcc = max(nx.weakly_connected_components(G), key=len)
print(f'En büyük WCC boyutu: {len(largest_wcc)} node ({100*len(largest_wcc)/G.number_of_nodes():.1f}%)')

# En büyük strongly connected component boyutu
largest_scc = max(nx.strongly_connected_components(G), key=len)
print(f'En büyük SCC boyutu: {len(largest_scc)} node ({100*len(largest_scc)/G.number_of_nodes():.1f}%)')

## 3. Centrality Ölçümleri

Her metrik hesaplandıktan sonra `./cache/` klasörüne kaydedilir.
Sonraki çalıştırmalarda cache'ten okunur.

Hesaplanacak metrikler:
- **Degree Centrality**
- **Betweenness Centrality** (yavaş – dikkat!)
- **Closeness Centrality**
- **PageRank** (eigenvector yerine)

In [ ]:
def compute_or_load(name, compute_fn):
    """İsim verilen metriği hesapla veya cache'ten yükle."""
    path = os.path.join(CACHE_DIR, f'{name}.pkl')
    if os.path.exists(path):
        print(f'  ✓ {name} cache\'ten yükleniyor…')
        with open(path, 'rb') as f:
            return pickle.load(f)
    print(f'  ⏳ {name} hesaplanıyor…')
    result = compute_fn()
    with open(path, 'wb') as f:
        pickle.dump(result, f)
    print(f'  ✓ {name} hesaplandı ve kaydedildi.')
    return result

In [ ]:
# Degree centrality (directed graf üzerinde)
degree_cent = compute_or_load('degree_centrality',
    lambda: nx.degree_centrality(G))

# Betweenness centrality (undirected üzerinde – daha hızlı)
betweenness_cent = compute_or_load('betweenness_centrality',
    lambda: nx.betweenness_centrality(G_und))

# Closeness centrality (undirected üzerinde)
closeness_cent = compute_or_load('closeness_centrality',
    lambda: nx.closeness_centrality(G_und))

# PageRank (directed graf üzerinde)
pagerank_cent = compute_or_load('pagerank',
    lambda: nx.pagerank(G))

In [ ]:
def centrality_stats(name, cent_dict):
    vals = np.array(list(cent_dict.values()))
    return {
        'Metric': name,
        'Min': vals.min(),
        'Max': vals.max(),
        'Mean': vals.mean(),
        'Median': np.median(vals),
        'Std': vals.std()
    }

import pandas as pd
stats = pd.DataFrame([
    centrality_stats('Degree Centrality', degree_cent),
    centrality_stats('Betweenness Centrality', betweenness_cent),
    centrality_stats('Closeness Centrality', closeness_cent),
    centrality_stats('PageRank', pagerank_cent),
])
print('Centrality İstatistikleri:')
print(stats.to_string(index=False))

In [ ]:
centralities = [
    ('Degree Centrality', degree_cent, '#2196F3'),
    ('Betweenness Centrality', betweenness_cent, '#4CAF50'),
    ('Closeness Centrality', closeness_cent, '#FF9800'),
    ('PageRank', pagerank_cent, '#9C27B0'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, (name, cent_dict, color) in enumerate(centralities):
    ax = axes[idx // 2][idx % 2]
    vals = list(cent_dict.values())
    ax.hist(vals, bins=80, color=color, edgecolor='black', alpha=0.85)
    ax.set_title(f'{name} Dağılımı', fontsize=13, fontweight='bold')
    ax.set_xlabel(name, fontsize=11)
    ax.set_ylabel('Frekans', fontsize=11)
    ax.axvline(np.mean(vals), color='red', linestyle='--',
               label=f'Mean: {np.mean(vals):.6f}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Centrality Dağılım Histogramları', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Her metrik için en yüksek 10 node
for name, cent_dict in [('Degree', degree_cent), ('Betweenness', betweenness_cent),
                         ('Closeness', closeness_cent), ('PageRank', pagerank_cent)]:
    top10 = sorted(cent_dict.items(), key=lambda x: x[1], reverse=True)[:10]
    print(f'\n=== Top-10 {name} Centrality ===')
    for rank, (nid, val) in enumerate(top10, 1):
        title = G.nodes[nid].get('title', nid)
        print(f'  {rank:2d}. {title}  ({val:.6f})')

## 4. Community Detection

### 4.1 Louvain Algoritması
### 4.2 Infomap Algoritması

Her iki algoritma için:
- Modularity
- Community sayısı
- Min / Max / Mean / Median community boyutu
- Standart sapma
- Community boyut dağılımı histogramı

### 4.1 Louvain Algoritması

In [ ]:
import community as community_louvain

def run_louvain():
    partition = community_louvain.best_partition(G_und)
    mod = community_louvain.modularity(partition, G_und)
    return partition, mod

louvain_result = compute_or_load('louvain_partition', run_louvain)
louvain_partition, louvain_modularity = louvain_result

print(f'Louvain Modularity: {louvain_modularity:.6f}')

In [ ]:
def community_stats(partition, algo_name):
    """Partition dict → istatistikler + histogram"""
    comm_sizes = Counter(partition.values())
    sizes = np.array(list(comm_sizes.values()))
    n_communities = len(comm_sizes)

    print(f'\n=== {algo_name} Community İstatistikleri ===')
    print(f'  Community sayısı : {n_communities}')
    print(f'  Min boyut        : {sizes.min()}')
    print(f'  Max boyut        : {sizes.max()}')
    print(f'  Mean boyut       : {sizes.mean():.2f}')
    print(f'  Median boyut     : {np.median(sizes):.1f}')
    print(f'  Std sapma        : {sizes.std():.2f}')

    # Histogram
    plt.figure(figsize=(10, 5))
    plt.hist(sizes, bins=max(20, n_communities // 2), color='#00BCD4', edgecolor='black', alpha=0.85)
    plt.title(f'{algo_name} – Community Boyut Dağılımı', fontsize=14, fontweight='bold')
    plt.xlabel('Community Boyutu (node sayısı)', fontsize=12)
    plt.ylabel('Frekans', fontsize=12)
    plt.axvline(sizes.mean(), color='red', linestyle='--', label=f'Mean: {sizes.mean():.1f}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return {'n_communities': n_communities, 'sizes': sizes}

louvain_stats = community_stats(louvain_partition, 'Louvain')

### 4.2 Infomap Algoritması

In [ ]:
import infomap

def run_infomap():
    # Infomap undirected graf üzerinde çalışır
    im = infomap.Infomap('--two-level --undirected')
    # Node'lara integer mapping gerekli
    node_to_idx = {n: i for i, n in enumerate(G_und.nodes())}
    idx_to_node = {i: n for n, i in node_to_idx.items()}
    for u, v in G_und.edges():
        im.add_link(node_to_idx[u], node_to_idx[v])
    im.run()

    # Partition oluştur
    partition = {}
    for node_id in im.tree:
        if node_id.is_leaf:
            partition[idx_to_node[node_id.node_id]] = node_id.module_id

    # Modularity hesapla (NetworkX kullanarak)
    communities_sets = {}
    for nid, comm_id in partition.items():
        communities_sets.setdefault(comm_id, set()).add(nid)
    comm_list = list(communities_sets.values())
    mod = nx.community.modularity(G_und, comm_list)
    return partition, mod

infomap_result = compute_or_load('infomap_partition', run_infomap)
infomap_partition, infomap_modularity = infomap_result

print(f'Infomap Modularity: {infomap_modularity:.6f}')

In [ ]:
infomap_stats = community_stats(infomap_partition, 'Infomap')

### 4.3 Louvain vs Infomap Karşılaştırması

In [ ]:
comp = pd.DataFrame([
    {
        'Algoritma': 'Louvain',
        'Modularity': round(louvain_modularity, 6),
        'Community Sayısı': louvain_stats['n_communities'],
        'Min Boyut': louvain_stats['sizes'].min(),
        'Max Boyut': louvain_stats['sizes'].max(),
        'Mean Boyut': round(louvain_stats['sizes'].mean(), 2),
        'Median Boyut': round(float(np.median(louvain_stats['sizes'])), 1),
        'Std': round(louvain_stats['sizes'].std(), 2),
    },
    {
        'Algoritma': 'Infomap',
        'Modularity': round(infomap_modularity, 6),
        'Community Sayısı': infomap_stats['n_communities'],
        'Min Boyut': infomap_stats['sizes'].min(),
        'Max Boyut': infomap_stats['sizes'].max(),
        'Mean Boyut': round(infomap_stats['sizes'].mean(), 2),
        'Median Boyut': round(float(np.median(infomap_stats['sizes'])), 1),
        'Std': round(infomap_stats['sizes'].std(), 2),
    },
])

print('\n=== Louvain vs Infomap Karşılaştırması ===')
print(comp.to_string(index=False))